**Note: For this assignment, you may only use standard Python and the `re` (Regular Expression) module. Advanced libraries such as NumPy, Pandas are not permitted**

## Exercises 1

Use `re.search` to find whether a string contains a phone number. The pattern that you write should detect a phone number in the following strings.  
```
"Call me at 382-384-3840."  
"my number is (510) 849-3519. Call me!"
```  
And not find a match in the following strings. 
```
"my number is 510-849-35192"  
"here’s my number: 510-849.3519"
``` 
Consider making your own tests as well  

In [1]:
import re

In [2]:
# YOUR CODE HERE
texts = [
    "Call me at 382-384-3840.",
    "my number is (510) 849-3519. Call me!",
    "my number is 510-849-35192",
    "here’s my number: 510-849.3519",
    # Custom test case
    "Another test: 111-222-3333 is valid.",
    "No match here: 12-34-5678"
]

# pattern: either 123-456-7890 or (123) 456-7890
pattern = r'(?:\d{3}-|\(\d{3}\)\s)\d{3}-\d{4}\b'

for text in texts:
    match = re.search(pattern, text)
    print(f"Text: '{text.strip()}'")
    if match:
        print(f"=> Match found: {match.group()}")
    else:
        print("=> No match")

Text: 'Call me at 382-384-3840.'
=> Match found: 382-384-3840
Text: 'my number is (510) 849-3519. Call me!'
=> Match found: (510) 849-3519
Text: 'my number is 510-849-35192'
=> No match
Text: 'here’s my number: 510-849.3519'
=> No match
Text: 'Another test: 111-222-3333 is valid.'
=> Match found: 111-222-3333
Text: 'No match here: 12-34-5678'
=> No match


## Exercise 2

Use `re.sub` to alter the string below so that the dates have a common format that uses a dash for the day, month, and year separator.  
```
03/12/2018, 03.13.18, 03/14/2018, 03:15:2018
```

In [3]:
# YOUR CODE HERE
text = "03/12/2018, 03.13.18, 03/14/2018, 03:15:2018"

# Normalize all separators to /
normalized_text = re.sub(r'[./:]', '/', text)

# Convert MM/DD/YYYY or MM/DD/YY to DD/MM/YYYY
def mmdd_to_ddmm(match):
    mm, dd, yy = match.groups()
    # If year has 2 digits, convert to 4 digits
    if len(yy) == 2:
        yy = '20' + yy
    return f"{dd}/{mm}/{yy}"

# Regex pattern: MM/DD/YYYY or MM/DD/YY
pattern = r'(\d{2})/(\d{2})/(\d{2,4})'
result = re.sub(pattern, mmdd_to_ddmm, normalized_text)

print(f"Original: {text}")
print(f"Converted (DD/MM/YYYY): {result}")

Original: 03/12/2018, 03.13.18, 03/14/2018, 03:15:2018
Converted (DD/MM/YYYY): 12/03/2018, 13/03/2018, 14/03/2018, 15/03/2018


## Exercise 3

Consider the first five sentences of the novel “Little Women” below. Extract the spoken dialog from each sentence.

In [4]:
text = '''
"Christmas won't be Christmas without any presents," grumbled Jo, lying on the rug.
"It's so dreadful to be poor!" sighed Meg, looking down at her old dress.
"I don't think it's fair for some girls to have plenty of pretty things, and other girls nothing at all," added little Amy, with an injured sniff.
"We've got Father and Mother, and each other," said Beth contentedly from her corner.
The four young faces on which the firelight shone brightened at the cheerful words, but darkened again as Jo said sadly, "We haven't got Father, and shall not have him for a long time."
'''
print(text)


"Christmas won't be Christmas without any presents," grumbled Jo, lying on the rug.
"It's so dreadful to be poor!" sighed Meg, looking down at her old dress.
"I don't think it's fair for some girls to have plenty of pretty things, and other girls nothing at all," added little Amy, with an injured sniff.
"We've got Father and Mother, and each other," said Beth contentedly from her corner.
The four young faces on which the firelight shone brightened at the cheerful words, but darkened again as Jo said sadly, "We haven't got Father, and shall not have him for a long time."



In [5]:
# YOUR CODE HERE
pattern = r'"[^"]+"'
spoken_dialog = re.findall(pattern, text)

# Delete the punctuation at the end of sentence (, ! ...)
spoken_dialog = [re.sub(r'([,.!?])"', '"', line) for line in spoken_dialog]

for line in spoken_dialog:
    print(line)

"Christmas won't be Christmas without any presents"
"It's so dreadful to be poor"
"I don't think it's fair for some girls to have plenty of pretty things, and other girls nothing at all"
"We've got Father and Mother, and each other"
"We haven't got Father, and shall not have him for a long time"


## Exercise 4

In this exercise, you you working with ```email_test.txt``` file (attached), using Regular Expression.\
`Original Dataset: https://www.kaggle.com/datasets/rtatman/fraudulent-email-corpus`

In [6]:
# YOUR CODE HERE: open file
try:
    with open("email_test.txt", "r", encoding="utf-8") as f:
        full_text = f.read()
except FileNotFoundError:
    print("ERROR: 'email_test.txt' not found. Cannot run Exercise 4.")
    emails = [] 

emails = re.split(r'(?=^From r )', full_text, flags=re.MULTILINE | re.IGNORECASE)
emails = [e.strip() for e in emails if e.strip()]

print(f"Total emails: {len(emails)}")

Total emails: 1330


#### Simple Fraudulent email detection

1. Count how many emails contain urgency-related words (URGENT, IMMEDIATELY, QUICK, ASSISTANCE, CONFIDENTIAL). 
Calculate what percentage of emails use these tactics.

In [7]:
# YOUR CODE HERE
urgent_pattern = ["urgent", "immediately", "quick", "assistance", "confidential"]

emails_with_urgency = 0

for mail in emails:
    text = mail.lower()
    for term in urgent_pattern:
        if re.search(rf"{term}", text):
            emails_with_urgency += 1
            break  

ratio = (emails_with_urgency / len(emails) * 100) if emails else 0

print(f"Emails containing urgency indicators: {emails_with_urgency}")
print(f"Percentage of emails using urgency language: {ratio:.2f}%")

Emails containing urgency indicators: 1184
Percentage of emails using urgency language: 89.02%


2. Find all mentions of money amounts in the email bodies (e.g., `US$25M`, `$100,000.00`, `USD$31,000,000.00`). Calculate:
- Total number of money mentions across all emails
- The largest amount mentioned
- The smallest amount mentioned
- Average amount per email

In [8]:
# YOUR CODE HERE
money_pattern = r'(?:US\$|USD\$|\$)\s*([\d,]+(?:\.\d{2})?)(?:\s*M|B)?' 

def clean_and_convert_money(s):
    """Làm sạch chuỗi tiền (xóa dấu phẩy, xử lý M/B) và chuyển đổi thành số thực."""
    # Kiểm tra M (Millions) hoặc B (Billions) và điều chỉnh hệ số nhân
    multiplier = 1.0
    s_upper = s.upper()
    if s_upper.endswith('M'):
        multiplier = 1000000.0
        s = s[:-1]
    elif s_upper.endswith('B'):
        multiplier = 1000000000.0
        s = s[:-1]

    try:
        # Loại bỏ dấu phẩy để chuyển đổi
        s_cleaned = s.replace(',', '')
        # Loại bỏ ký hiệu USD nếu nó vẫn còn trong chuỗi (trường hợp group 1 có thể bao gồm nó)
        s_cleaned = s_cleaned.replace('$', '').replace('US', '').replace('USD', '')
        return float(s_cleaned) * multiplier
    except ValueError:
        return 0.0

all_amounts = []

for email in emails:
    # re.findall trả về nội dung của nhóm bắt giữ (capturing group 1)
    mentions = re.findall(money_pattern, email, flags=re.IGNORECASE)
    
    for amount_str in mentions:
        amount_float = clean_and_convert_money(amount_str)
        if amount_float > 0.0:
            all_amounts.append(amount_float)

total_mentions = len(all_amounts)
largest_amount = max(all_amounts) if all_amounts else 0
smallest_amount = min(all_amounts) if all_amounts else 0
average_amount = sum(all_amounts) / total_mentions if total_mentions else 0

print(f"Total number of money mentions (USD only): {total_mentions}")
print(f"The largest amount mentioned (USD only): ${largest_amount:,.2f}")
print(f"The smallest amount mentioned (USD only): ${smallest_amount:,.2f}")
print(f"Average amount per email (USD only): ${average_amount:,.2f}")

Total number of money mentions (USD only): 1983
The largest amount mentioned (USD only): $1,800,000,000.00
The smallest amount mentioned (USD only): $0.28
Average amount per email (USD only): $17,238,459.99


3. Extract all mentions of deaths or deceased persons (e.g., "late father", "died", "deceased", "death of").\
   What percentage of emails use death as part of their story?

In [9]:
# YOUR CODE HERE
death_pattern = re.compile(
    r'''
    (?:                                   # groups of clusters that may appear
        \blate(?:\s+(?:mr|mrs|ms|dr|prof|father|mother|husband|wife|
                     son|daughter|uncle|aunt))?\b   # for example: late father, late Mr Smith
      | \b(?:died|deceased|dead|death|killed|murdered|perished|
             victim|shot|buried|funeral|memorial)\b
      | \bloss\s+(?:of|my)\b              # 'loss of my father'
      | \bpassed\s+away\b
      | \blost\s+(?:their|there)\s+lives\b
      | \bfound\s+dead\b
    )
    ''',
    re.IGNORECASE | re.VERBOSE
)

emails_with_death = 0

for mail in emails:
    if death_pattern.search(mail):
        emails_with_death += 1

ratio = (emails_with_death / len(emails) * 100) if emails else 0

print(f"Emails mentioning death-related stories: {emails_with_death}")
print(f"Percentage of emails use death as part of their story: {ratio:.2f}%")

Emails mentioning death-related stories: 845
Percentage of emails use death as part of their story: 63.53%


4. Many emails mention percentage splits of money (e.g., "70% for us", "20% for you", "10% for expenses"). \
   Extract all percentage distributions and identify the most common split pattern offered to recipients.

   Example: most common split patterns:\
   70% - 20% - 10%: appears 15 times\
   75% - 20% - 5%: appears 12 times\
    60% - 30% - 10%: appears 8 times\
    80% - 15% - 5%: appears 6 times\
    55% - 30% - 10% - 5%: appears 4 times

In [10]:
# YOUR CODE HERE
percentage_pattern = re.compile(r'\b([1-9]?\d)\s*(?:%|percent)', re.IGNORECASE)

split_patterns = []

for email in emails:
    matches = percentage_pattern.findall(email.lower())
    total = 0.0
    valid_split = []
    for p in matches:
        try:
            value = float(p)
            if 0 < value <= 100 and total + value <= 100:
                total += value
                valid_split.append(value)
        except ValueError:
            continue
    if valid_split:
        split_patterns.append(tuple(valid_split))

pattern_freq = {}
for sp in split_patterns:
    key = tuple(sorted(sp, reverse=True))
    pattern_freq[key] = pattern_freq.get(key, 0) + 1

pattern_freq_sorted = dict(sorted(pattern_freq.items(), key=lambda x: x[1], reverse=True))

print("List percentage split patterns:")
for pattern, count in pattern_freq_sorted.items():
    pattern_str = ' - '.join(f"{p}%" for p in pattern)
    print(f"  {pattern_str}: appears {count} times")

List percentage split patterns:
  60.0% - 30.0% - 10.0%: appears 100 times
  70.0% - 25.0% - 5.0%: appears 82 times
  20.0%: appears 64 times
  70.0% - 20.0% - 10.0%: appears 52 times
  30.0%: appears 52 times
  75.0% - 20.0% - 5.0%: appears 45 times
  25.0%: appears 44 times
  10.0%: appears 41 times
  60.0% - 35.0% - 5.0%: appears 37 times
  80.0% - 20.0%: appears 31 times
  65.0% - 30.0% - 5.0%: appears 30 times
  60.0% - 40.0%: appears 27 times
  30.0% - 10.0%: appears 18 times
  70.0% - 30.0%: appears 18 times
  15.0%: appears 17 times
  65.0% - 25.0% - 10.0%: appears 14 times
  75.0% - 25.0%: appears 13 times
  20.0% - 10.0%: appears 13 times
  15.0% - 5.0%: appears 12 times
  25.0% - 5.0%: appears 10 times
  20.0% - 5.0%: appears 9 times
  99.0%: appears 9 times
  40.0%: appears 8 times
  25.0% - 25.0%: appears 8 times
  80.0% - 15.0% - 5.0%: appears 8 times
  25.0% - 10.0%: appears 7 times
  50.0% - 40.0% - 10.0%: appears 7 times
  65.0% - 20.0% - 10.0% - 5.0%: appears 7 times


5. Create a "scam score" for each email based on:\
    Urgency keywords (1 point each)\
    Money mentions (2 points each)\
    Percentages offered (1 point each)\
    Death mentions (1 point)\
    ALL CAPS usage (1 point if >20% of text)

    ***Rank the top 10 highest-scoring emails*** 

In [11]:
# YOUR CODE HERE
urgent_pattern = ["urgent", "immediately", "quick", "assistance", "confidential"]
money_pattern_str = r'(?:US\$|USD\$|\$)\s*([\d,]+(?:\.\d{2})?)(?:\s*M|B)?' 
percentage_pattern_str = r'\b([1-9]?\d)\s*(?:%|percent)'
death_pattern_str = r'''
    (?:
        \blate(?:\s+(?:mr|mrs|ms|dr|prof|father|mother|husband|wife|
                     son|daughter|uncle|aunt))?\b
      | \b(?:died|deceased|dead|death|killed|murdered|perished|
             victim|shot|buried|funeral|memorial)\b
      | \bloss\s+(?:of|my)\b
      | \bpassed\s+away\b
      | \blost\s+(?:their|there)\s+lives\b
      | \bfound\s+dead\b
    )
    '''

# compile all patterns outside the loop 
urgent_regex = re.compile(r'\b(?:' + '|'.join(urgent_pattern) + r')\b', re.IGNORECASE)
money_regex = re.compile(money_pattern_str, re.IGNORECASE)
percentage_regex = re.compile(percentage_pattern_str, re.IGNORECASE)
death_regex = re.compile(death_pattern_str, re.IGNORECASE | re.VERBOSE)


def all_caps_point(text, threshold=0.2):
    """Returns 1 if the ratio of uppercase letters to all letters exceeds the threshold."""
    letters = [c for c in text if c.isalpha()]        
    if not letters:
        return 0
    caps_count = sum(1 for c in letters if c.isupper())
    ratio = caps_count / len(letters)
    return 1 if ratio > threshold else 0

scam_scores = []

# Assuming 'emails' is loaded and contains the email data
if 'emails' in locals() and emails:
    for i, email in enumerate(emails):
        text = email.lower()

        # Urgency keywords -> +1 per match
        urgency_points = len(urgent_regex.findall(text))

        # Money mentions -> +2 per match
        money_points = 2 * len(money_regex.findall(email)) 

        # Percentages offered -> +1 per match
        percent_points = len(percentage_regex.findall(text))

        # Death mentions -> +1 if any
        # Use .search() for a presence check
        death_points = 1 if death_regex.search(email) else 0 

        # ALL CAPS usage (>20%) -> +1
        caps_points = all_caps_point(email)

        total_score = urgency_points + money_points + percent_points + death_points + caps_points
        scam_scores.append((i, total_score))

    # Sort & print top 10
    scam_scores.sort(key=lambda x: x[1], reverse=True)

    if len(scam_scores) >= 10:
        threshold_score = scam_scores[9][1]
    else:
        threshold_score = scam_scores[-1][1] if scam_scores else 0

    print("Top 10 highest-scoring emails (including ties):")
    for i, score in scam_scores:
        if score >= threshold_score and score > 0:
            print(f"Email index {i}: score = {score}")
        elif score < threshold_score and score > 0:
            break 
        elif score == 0:
            break
else:
    print("Error: 'emails' list is not available or empty. Cannot calculate scam scores.")

Top 10 highest-scoring emails (including ties):
Email index 133: score = 34
Email index 590: score = 32
Email index 243: score = 30
Email index 393: score = 29
Email index 415: score = 27
Email index 459: score = 27
Email index 460: score = 27
Email index 461: score = 27
Email index 66: score = 26
Email index 104: score = 26
Email index 944: score = 26


6. Identify emails that appear to be duplicates or near-duplicates (same sender, similar subject, sent within 24 hours). How many duplicate emails exist?

In [12]:
# YOUR CODE HERE
sender_pattern = re.compile(r'From:.*<(.*?)>', re.IGNORECASE)
subject_pattern = re.compile(r'Subject: (.*)', re.IGNORECASE)
date_pattern_extract = re.compile(r'Date: (.*)', re.IGNORECASE)

senders = []
subjects = []
dates = [] 

for email in emails:
    # Sender
    sender_match = sender_pattern.search(email)
    senders.append(sender_match.group(1).strip() if sender_match else "UNKNOWN_SENDER")
    
    # Subject
    subject_match = subject_pattern.search(email)
    subjects.append(subject_match.group(1).strip() if subject_match else "NO_SUBJECT")
    
    # Date
    date_match = date_pattern_extract.search(email)
    dates.append(date_match.group(1).strip() if date_match else "UNKNOWN_DATE")

def hours_difference_approx(date1, date2):
    """
    Approximates time difference by extracting the day number from the date string.
    Returns 0 if day numbers are the same, 25 ( > 24) otherwise.
    This is a simplification due to library constraints.
    """
    day_regex = re.compile(r'\b(\d{1,2})\s+[A-Za-z]{3}')
    
    day1_match = day_regex.search(date1)
    day2_match = day_regex.search(date2)
    
    day1 = int(day1_match.group(1)) if day1_match else -1
    day2 = int(day2_match.group(1)) if day2_match else -2
    
    # within 24 hours is simplified to "same day" (or a difference of 1, e.g., 30 vs 1)
    # a true check is impossible without external libraries. We'll use "same day number."
    return 0 if day1 == day2 and day1 > 0 else 25 # 0 means "within 24h", 25 means "outside 24h"

def simple_similarity(a, b):
    """Calculates the ratio of common characters in the shorter string to the length of the longer string."""
    a, b = a.lower(), b.lower()
    min_len = min(len(a), len(b))
    match = 0
    for i in range(min_len):
        if a[i] == b[i]:
            match += 1
    return match / max(len(a), len(b)) if max(len(a), len(b)) > 0 else 0

duplicate_pairs = []
n = len(senders)
time_limit_hours = 24
similarity_threshold = 0.8

for i in range(n):
    for j in range(i + 1, n):
        # Same Sender (case-insensitive)
        same_sender = senders[i].strip().lower() == senders[j].strip().lower()
        
        # Similar Subject (>= 80% match)
        similar_subject = simple_similarity(subjects[i], subjects[j]) >= similarity_threshold
        
        # Sent Within 24 Hours (Approximation used)
        within_24h = hours_difference_approx(dates[i], dates[j]) <= time_limit_hours
        
        if same_sender and similar_subject and within_24h:
            duplicate_pairs.append((i, j))

print(f"Number of duplicate or near-duplicate email pairs: {len(duplicate_pairs)}")
if duplicate_pairs:
    print("10 example duplicate pairs (index i, j):")
    for pair in duplicate_pairs[:10]:
        print(f"  {pair}")

Number of duplicate or near-duplicate email pairs: 350
10 example duplicate pairs (index i, j):
  (2, 3)
  (16, 17)
  (18, 19)
  (24, 25)
  (46, 47)
  (57, 73)
  (68, 71)
  (69, 70)
  (76, 77)
  (93, 94)
